In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 데이터 불러오기
df = pd.read_csv('../data/processed/df_merged.csv', encoding='cp949')
df.head()

In [ ]:
print(df.isna().sum())
df.dropna(inplace=True)

In [ ]:
print(df.isna().sum())

In [ ]:
df.columns

In [ ]:
# 비율 계산
unique_users = df[['idUser', 'MemberYN']].drop_duplicates()
member_ratio = unique_users['MemberYN'].value_counts(normalize=True)
member_count = unique_users['MemberYN'].value_counts()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# 고객 수
sns.countplot(data=unique_users, x='MemberYN', ax=ax[0])
ax[0].set_title('멤버십 고객 수')
ax[0].set_ylabel('명')

for i, v in enumerate(member_count):
    ax[0].text(i, v, f'{v:,}', ha='center', va='bottom')

# 비율
member_ratio.plot(kind='bar', ax=ax[1])
ax[1].set_title('멤버십 비율')
ax[1].set_ylabel('비율(%)')

for i, v in enumerate(member_ratio):
    ax[1].text(i, v, f'{v:.2%}', ha='center', va='bottom')

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 연령대별 멤버십 비율
unique_users = df[['idUser','AgeGroup','Gender','MemberYN']].drop_duplicates()
age_member = pd.crosstab(unique_users['AgeGroup'], unique_users['MemberYN'], normalize='index')

age_member.plot(kind='bar', stacked=True, figsize=(10,5))
plt.title('연령대에 따른 멤버십 비율')
plt.xlabel('연령대')
plt.ylabel('비율')
plt.xticks(rotation=0)
plt.show()

# 성별에 따른 멤버십 비율
gender_member = pd.crosstab(unique_users['Gender'], unique_users['MemberYN'], normalize='index')

gender_member.plot(kind='bar', stacked=True, figsize=(6,4))
plt.title('성별에 따른 멤버십 비율')
plt.xticks(rotation=0)
plt.xlabel('성별')
plt.ylabel('비율')
plt.show()

In [ ]:
# 멤버십 여부에 따른 유저별 평균 구매 금액
user_order_sum = df.groupby(['idUser','MemberYN','idOrder'])['Price'].sum().reset_index()
user_total_avg = user_order_sum.groupby(['idUser','MemberYN'])['Price'].mean().reset_index()

sns.boxplot(data=user_total_avg, x='MemberYN', y='Price')
plt.title('멤버십 여부에 따른 유저별 평균 구매 금액')
plt.xlabel('멤버십 여부')
plt.ylabel('평균 구매 가격')
plt.show()

In [ ]:
# 멤버십 여부에 따른 1회 주문 금액
order_price = df.groupby(['idOrder', 'MemberYN'])['Price'].sum().reset_index()

sns.boxplot(data=order_price, x='MemberYN', y='Price')
plt.title('멤버십 여부에 따른 1회 주문 금액')
plt.xlabel('멤버십 여부')
plt.ylabel('총 구매 가격')
plt.show()

In [ ]:
sns.kdeplot(data=order_price, x='Price', hue='MemberYN', fill=True)
plt.title('멤버십 여부에 따른 1회 주문 금액 분포')
plt.xlim(0, 50000)  # 극단값 제거하고 확대 가능
plt.xlabel('총 구매 금액')
plt.show()

In [ ]:
# 멤버십 여부에 따른 지연 여부
unique_orders = df[['idOrder','MemberYN','IsLate']].drop_duplicates()
late_ratio = pd.crosstab(unique_orders['IsLate'], unique_orders['MemberYN'], normalize='index')

late_ratio.plot(kind='bar', stacked=True)
plt.title('멤버십 여부에 따른 배송 지연')
plt.xticks(rotation=0)
plt.xlabel('배송 지연')
plt.ylabel('비율')
plt.show()

In [ ]:
# 멤버십 여부에 따른 주문 시간 차이
hour_member = pd.crosstab(df['OrderHour'], df['MemberYN'], normalize='index')

hour_member.plot(figsize=(12,5))
plt.title('멤버십 여부에 따른 주문 시간 차이')
plt.ylabel('비율')
plt.show()
# 패턴에 유의한 차이 없음

In [ ]:
# 멤버십 여부에 따른 주문 품목 차이(대분류명 기준)
# item_member = pd.crosstab(df['ItemLargeName'], df['MemberYN'], normalize='index')
user_item = df[['idUser','MemberYN','ItemLargeName']].drop_duplicates()
item_member = pd.crosstab(user_item['ItemLargeName'], user_item['MemberYN'], normalize='index')

item_member.sort_values(by='Y', ascending=False).plot(
    kind='barh', stacked=True, figsize=(10,8))

plt.title('상품 분류별 차이')
plt.xlabel('비율')
plt.ylabel('상품 대분류명')
plt.show()

In [ ]:
# 멤버십 여부에 따른 주문 시간대 상관분석
pivot_y = df[df['MemberYN']=='Y'].pivot_table(
    index='OrderWeekday', columns='OrderHour', values='idOrder', aggfunc='count')

pivot_n = df[df['MemberYN']=='N'].pivot_table(
    index='OrderWeekday', columns='OrderHour', values='idOrder', aggfunc='count')

fig, ax = plt.subplots(1,2, figsize=(16,6))

sns.heatmap(pivot_y, ax=ax[0], cmap='Blues')
ax[0].set_xlabel('주문 시간')
ax[0].set_ylabel('주문 요일')
ax[0].set_title('Members (Y)')

sns.heatmap(pivot_n, ax=ax[1], cmap='Reds')
ax[1].set_title('Non-Members (N)')
ax[1].set_xlabel('주문 시간')
ax[1].set_ylabel('주문 요일')
plt.show()
# 패턴 차이 없음

In [ ]:
# 멤버십 여부에 따른 이용자 재구매율
user_order_cnt = df.groupby(['idUser', 'MemberYN'])['idOrder'].nunique().reset_index()

sns.kdeplot(data=user_order_cnt, x='idOrder', hue='MemberYN', fill=True)
plt.title('재구매')
plt.xlabel('유저별 주문 횟수')
plt.ylabel('밀도')
plt.show()
user_order_cnt
# 패턴 유사

In [ ]:
rfm = df.groupby(['idUser', 'MemberYN']).agg({
    'OrderDT': lambda x: (pd.to_datetime('today') - pd.to_datetime(x).max()).days,
    'idOrder': 'nunique',
    'Price': 'sum'
}).reset_index()

rfm.columns = ['idUser', 'MemberYN', 'Recency', 'Frequency', 'Monetary']

sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='MemberYN')
plt.title('RFM 패턴')
plt.show()

In [ ]:
for label, grp in user_order_cnt.groupby('MemberYN'):
    sorted_vals = np.sort(grp['idOrder'])
    yvals = np.arange(len(sorted_vals)) / float(len(sorted_vals))
    plt.plot(sorted_vals, yvals, label=label)

plt.legend()
plt.title('주문 횟수 누적 분포')
plt.xlabel('주문 횟수')
plt.ylabel('CDF')
plt.show()

In [ ]:
# 멤버십 여부에 따른 리프트값
late_lift = pd.crosstab(df['IsLate'], df['MemberYN'], normalize='columns')

lift = late_lift['Y'] / late_lift['N']
lift.plot(kind='bar')
plt.title('멤버십 여부에 따른 Lift값')
plt.axhline(1, color='red', linestyle='--')
plt.xticks(rotation=0)
plt.xlabel('배송 지연')
plt.show()
# 멤버십에 가입한 경우 리프트값이 1을 넘어가기는 하나, 실제 지연율 차이는 0.13%로 미미한 수준임.

In [ ]:
late_ratio = pd.crosstab(df['IsLate'], df['MemberYN'], normalize='columns')

y_rate = late_ratio.loc[True, 'Y']
n_rate = late_ratio.loc[True, 'N']

diff = (y_rate - n_rate) * 100

print(f"회원 지연률: {y_rate:.4f}")
print(f"비회원 지연률: {n_rate:.4f}")
print(f"차이: {diff:.2f}%p")

In [ ]:
# 연령대별 대분류 구매 건수 집계
age_category = df.groupby(['AgeGroup', 'ItemLargeCode'])['idOrder'].count().reset_index()

# 연령대별 비율 계산
age_totals = age_category.groupby('AgeGroup')['idOrder'].sum().reset_index().rename(columns={'idOrder':'Total'})
age_category = age_category.merge(age_totals, on='AgeGroup')
age_category['Ratio'] = age_category['idOrder'] / age_category['Total']

# pivot 형태로 변환 (연령대 x 대분류)
pivot = age_category.pivot(index='AgeGroup', columns='ItemLargeCode', values='Ratio')
print(df['ItemLargeCode'].value_counts())
print(pivot)

# heatmap 시각화
plt.figure(figsize=(12,6))
sns.heatmap(pivot, annot=True, fmt=".2f")
plt.title('연령대별 대분류 구매 비율')
plt.xlabel('대분류')
plt.ylabel('연령대')
plt.show()
# 연령대별 차이 x